In [1]:
!pip install -q -U peft bitsandbytes

In [2]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

peft_model_id = "Tirendaz/mistral-7b-dolly-fine-tuned"

config = PeftConfig.from_pretrained(peft_model_id)

In [3]:
from transformers import BitsAndBytesConfig

# 4-bit quant config (önerilen)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4/P100 için genelde iyi
)

In [4]:
# Base modeli 4-bit yükle
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    return_dict=True,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path,
                                          padding_side = "right",
                                          add_eos_token = True)
tokenizer.pad_token = tokenizer.eos_token

In [6]:
fine_tuned_model = PeftModel.from_pretrained(model, peft_model_id)

In [7]:
from transformers import pipeline, logging

logging.set_verbosity(logging.CRITICAL)

pipe = pipeline(
    task="text-generation",
    model=fine_tuned_model,
    tokenizer=tokenizer,
    eos_token_id=model.config.eos_token_id,
    max_new_tokens=100)

In [8]:
prompt = """
What is a Python?  Here is some context: Python is a high-level, general-purpose programming language.
"""
pipe = pipeline(task="text-generation",
                model=fine_tuned_model,
                tokenizer=tokenizer,
                eos_token_id=model.config.eos_token_id,
                max_new_tokens=100)

result = pipe(f"<s>[INST] {prompt} [/INST]")
generated = result[0]['generated_text']
print(generated[generated.find('[/INST]')+8:])

а Python is a high-level, general-purpose programming language.  It is often compared to Perl, Ruby, or Tcl. Python was created in the late 1980s, and released in 1991.

Python is an object-oriented language, meaning that it is designed to work with objects. An object is an entity that contains data and methods that operate on that data. The most common objects in Python are classes, which are


In [9]:
prompt = """
Please summarize what Linkedin does. Here is some context: LinkedIn is a business and employment-focused social media platform
"""
pipe = pipeline(task="text-generation",
                model=fine_tuned_model,
                tokenizer=tokenizer,
                eos_token_id=model.config.eos_token_id,
                max_new_tokens=100)

result = pipe(f"<s>[INST] {prompt} [/INST]")
generated = result[0]['generated_text']
print(generated[generated.find('[/INST]')+8:])

LinkedIn is a business and employment-focused social media platform. It is mostly used for professional networking, and contains a large number of employee-employer connections. LinkedIn was launched on May 5, 2003, and is mainly used for professional networking.

LinkedIn was launched on May 5, 2003, and is mainly used for professional networking. It has more than 700 million registered members spread across 200
